# Encoding and Mixing Examples on Titanic Dataset

This notebook demonstrates encoding and mixing operations on:
1. **Binary features** (SibSp_bin, Parch_bin)
2. **Categorical features** (Pclass, Sex, Embarked)
3. **Numerical features** (Age, Fare)

Each example shows:
- Original data
- After encoding
- After mixing
- Combined pipeline (encode + mix)


In [ ]:
# Setup: Imports and Data Loading
import sys
from pathlib import Path

# Add project root to path
notebook_path = Path().resolve()
if notebook_path.name == 'notebooks':
    project_root = notebook_path.parent
else:
    project_root = notebook_path

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
from quagua.data_utils import load_titanic_data
from quagua.encoders import (
    # Encoders (irreversible transformations)
    ChaoticSystemEncoder,
    OrthogonalSubspaceEncoder,
    FastPrivacyEncoder,  # ✅ NEW: Fast privacy encoder (like FHE but faster)
    # Mixers (feature mixing)
    ClassicalXORMixer,
    QuantumEntanglementEncoder,
    QuantumWalkEncoder,
    # Pipeline
    EncodeThenMixPipeline,
)

np.random.seed(42)

print("="*80)
print("ENCODING AND MIXING EXAMPLES ON TITANIC DATASET")
print("="*80)

# Load Titanic data
X, y, feature_names = load_titanic_data(include_continuous=True)
print(f"\n📊 Loaded {X.shape[0]} samples with {X.shape[1]} features")
print(f"Features: {feature_names}")

# Identify feature types
binary_features = ['SibSp_bin', 'Parch_bin']
categorical_features = ['Pclass', 'Sex', 'Embarked']
numerical_features = ['Age', 'Fare']

print(f"\nFeature types:")
print(f"  Binary: {binary_features}")
print(f"  Categorical: {categorical_features}")
print(f"  Numerical: {numerical_features}")

# Extract a small sample for demonstration
sample_size = 10
X_sample = X[:sample_size].copy()

print("\n" + "="*80)
print("ORIGINAL DATA SAMPLE (5 rows)")
print("="*80)
df_sample = pd.DataFrame(X_sample, columns=feature_names)
display(df_sample.round(3))


## Example 1: Binary Features (SibSp_bin, Parch_bin)

Binary features are 0/1 values. We'll show:
1. Encoding with Chaotic System
2. Mixing with XOR Mixer
3. Combined pipeline


In [ ]:
# Extract binary features
binary_idx = [feature_names.index(f) for f in binary_features]
X_binary = X_sample[:, binary_idx]

print(f"Original binary features ({sample_size} samples, {len(binary_features)} features):")
display(pd.DataFrame(X_binary, columns=binary_features))


### 1.1 Encoding: Chaotic System (Logistic Map)

The chaotic encoder transforms binary values (0/1) into continuous values in [0,1] through chaotic dynamics.


In [ ]:
encoder_chaotic = ChaoticSystemEncoder(map_type='logistic', n_iterations=5)
X_binary_encoded = encoder_chaotic.fit_transform(X_binary)

print(f"After encoding: shape {X_binary_encoded.shape}")
print("Note: Binary values (0/1) are transformed to continuous values in [0,1]")
display(pd.DataFrame(X_binary_encoded, columns=[f"{f}_encoded" for f in binary_features]).round(3))


### 1.2 Mixing: Classical XOR Mixer

The XOR mixer combines features together using circular XOR/additive operations.


In [ ]:
mixer_xor = ClassicalXORMixer()
mixer_xor.fit(X_binary_encoded)
X_binary_mixed = mixer_xor.transform(X_binary_encoded)

print(f"After mixing: shape {X_binary_mixed.shape}")
print("Note: Features are mixed together (circular XOR operation)")
display(pd.DataFrame(X_binary_mixed, columns=[f"{f}_mixed" for f in binary_features]).round(3))


### 1.3 Pipeline: Encode then Mix

Combining encoding and mixing in a single pipeline.


In [ ]:
pipeline_binary = EncodeThenMixPipeline(
    ChaoticSystemEncoder(map_type='logistic', n_iterations=5),
    ClassicalXORMixer()
)
X_binary_pipeline = pipeline_binary.fit_transform(X_binary)

print(f"After pipeline: shape {X_binary_pipeline.shape}")
display(pd.DataFrame(X_binary_pipeline, columns=[f"{f}_final" for f in binary_features]).round(3))


## Example 2: Categorical Features (Pclass, Sex, Embarked)

Categorical features are already normalized to [0,1] range. We'll show:
1. Encoding with Orthogonal Subspace (expands dimensions)
2. Mixing with Quantum Entanglement
3. Combined pipeline


In [ ]:
# Extract categorical features
cat_idx = [feature_names.index(f) for f in categorical_features]
X_cat = X_sample[:, cat_idx]

print(f"Original categorical features ({sample_size} samples, {len(categorical_features)} features):")
print("Note: Already normalized to [0,1] range")
display(pd.DataFrame(X_cat, columns=categorical_features).round(3))


### 2.1 Encoding: Orthogonal Subspace Encoder

Projects each feature into a higher-dimensional orthogonal subspace (3D per feature).


In [ ]:
encoder_ortho = OrthogonalSubspaceEncoder(subspace_dim=3)
X_cat_encoded = encoder_ortho.fit_transform(X_cat)

print(f"After encoding: shape {X_cat.shape} → {X_cat_encoded.shape} (expansion: {X_cat_encoded.shape[1]/X_cat.shape[1]:.1f}x)")
print("Note: Each feature is projected into a 3D orthogonal subspace")
display(pd.DataFrame(X_cat_encoded, columns=[f"dim_{i}" for i in range(X_cat_encoded.shape[1])]).round(3))


### 2.2 Mixing: Quantum Entanglement Encoder

Non-linearly mixes features with quantum-inspired correlations.


In [ ]:
mixer_entangle = QuantumEntanglementEncoder(entanglement_strength=0.4)
mixer_entangle.fit(X_cat_encoded)
X_cat_mixed = mixer_entangle.transform(X_cat_encoded)

print(f"After mixing: shape {X_cat_mixed.shape}")
print("Note: Features are entangled (non-linearly mixed) with quantum-inspired correlations")
display(pd.DataFrame(X_cat_mixed, columns=[f"dim_{i}" for i in range(X_cat_mixed.shape[1])]).round(3))


### 2.3 Pipeline: Encode then Mix

Combining orthogonal encoding and quantum entanglement.


In [ ]:
pipeline_cat = EncodeThenMixPipeline(
    OrthogonalSubspaceEncoder(subspace_dim=3),
    QuantumEntanglementEncoder(entanglement_strength=0.4)
)
X_cat_pipeline = pipeline_cat.fit_transform(X_cat)

print(f"After pipeline: shape {X_cat_pipeline.shape}")
display(pd.DataFrame(X_cat_pipeline, columns=[f"dim_{i}" for i in range(X_cat_pipeline.shape[1])]).round(3))


## Example 3: Numerical Features (Age, Fare)

Numerical features are standardized (mean=0, std=1). We'll show:
1. Encoding with Chaotic System (Henon Map)
2. Mixing with Quantum Walk (expands dimensions)
3. Combined pipeline


In [ ]:
# Extract numerical features
num_idx = [feature_names.index(f) for f in numerical_features]
X_num = X_sample[:, num_idx]

print(f"Original numerical features ({sample_size} samples, {len(numerical_features)} features):")
print("Note: Already standardized (mean=0, std=1)")
display(pd.DataFrame(X_num, columns=numerical_features).round(3))


### 3.1 Encoding: Chaotic System (Henon Map)

Transforms continuous values through chaotic dynamics.


In [ ]:
encoder_henon = ChaoticSystemEncoder(map_type='henon', n_iterations=5)
X_num_encoded = encoder_henon.fit_transform(X_num)

print(f"After encoding: shape {X_num_encoded.shape}")
print("Note: Continuous values are transformed through chaotic dynamics")
display(pd.DataFrame(X_num_encoded, columns=[f"{f}_encoded" for f in numerical_features]).round(3))


### 3.2 Mixing: Quantum Walk Encoder

Features undergo quantum walk on a graph, creating expanded representation.


In [ ]:
mixer_walk = QuantumWalkEncoder(n_steps=3, graph_type='complete')
mixer_walk.fit(X_num_encoded)
X_num_mixed = mixer_walk.transform(X_num_encoded)

print(f"After mixing: shape {X_num.shape} → {X_num_mixed.shape} (expansion: {X_num_mixed.shape[1]/X_num.shape[1]:.1f}x)")
print("Note: Features undergo quantum walk on a graph, creating expanded representation")
display(pd.DataFrame(X_num_mixed, columns=[f"walk_{i}" for i in range(X_num_mixed.shape[1])]).round(3))


### 3.3 Pipeline: Encode then Mix

Combining chaotic encoding and quantum walk.


In [ ]:
pipeline_num = EncodeThenMixPipeline(
    ChaoticSystemEncoder(map_type='henon', n_iterations=5),
    QuantumWalkEncoder(n_steps=3, graph_type='complete')
)
X_num_pipeline = pipeline_num.fit_transform(X_num)

print(f"After pipeline: shape {X_num_pipeline.shape}")
display(pd.DataFrame(X_num_pipeline, columns=[f"final_{i}" for i in range(X_num_pipeline.shape[1])]).round(3))


## Example 5: Fast Privacy Encoder (NEW - Like FHE but Faster)

Fast Privacy Encoder is a fast, one-way privacy-preserving encoder inspired by FHE principles, but designed for speed and ML utility. It uses hash-based lookup tables for discrete values and polynomial feature expansion.

We'll show:
1. Encoding on Binary features
2. Encoding on Categorical features  
3. Encoding on Numerical features
4. Encoding on all features together


### 5.1 Fast Privacy Encoder on Binary Features

Fast Privacy Encoder uses hash-based lookup tables for discrete values - perfect for binary data!


In [ ]:
# Extract binary features
binary_idx = [feature_names.index(f) for f in binary_features]
X_binary_fast = X_sample[:, binary_idx]

print(f"Original binary features ({sample_size} samples, {len(binary_features)} features):")
display(pd.DataFrame(X_binary_fast, columns=binary_features))

# Fast Privacy Encoder
encoder_fast_binary = FastPrivacyEncoder(
    encoding_dim=4,
    polynomial_degree=1,  # Linear (fast, less expansion)
    noise_level=0.1,
    use_mixing=True,
    hash_seed=42
)
X_binary_fast_encoded = encoder_fast_binary.fit_transform(X_binary_fast)

print(f"\nAfter FastPrivacyEncoder:")
print(f"Shape: {X_binary_fast.shape} → {X_binary_fast_encoded.shape} (expansion: {X_binary_fast_encoded.shape[1]/X_binary_fast.shape[1]:.1f}x)")
print("Note: Hash-based encoding for discrete values + polynomial expansion + mixing")
display(pd.DataFrame(X_binary_fast_encoded, columns=[f"fast_{i}" for i in range(X_binary_fast_encoded.shape[1])]).round(3))


### 5.2 Fast Privacy Encoder on Categorical Features

Works excellently on categorical data using hash-based lookup tables.


In [ ]:
# Extract categorical features
cat_idx = [feature_names.index(f) for f in categorical_features]
X_cat_fast = X_sample[:, cat_idx]

print(f"Original categorical features ({sample_size} samples, {len(categorical_features)} features):")
display(pd.DataFrame(X_cat_fast, columns=categorical_features).round(3))

# Fast Privacy Encoder
encoder_fast_cat = FastPrivacyEncoder(
    encoding_dim=4,
    polynomial_degree=1,
    noise_level=0.1,
    use_mixing=True,
    hash_seed=42
)
X_cat_fast_encoded = encoder_fast_cat.fit_transform(X_cat_fast)

print(f"\nAfter FastPrivacyEncoder:")
print(f"Shape: {X_cat_fast.shape} → {X_cat_fast_encoded.shape} (expansion: {X_cat_fast_encoded.shape[1]/X_cat_fast.shape[1]:.1f}x)")
print("Note: Hash tables map each unique categorical value to encoding vector")
display(pd.DataFrame(X_cat_fast_encoded, columns=[f"fast_{i}" for i in range(X_cat_fast_encoded.shape[1])]).round(3))


### 5.3 Fast Privacy Encoder on Numerical Features

For continuous features, FastPrivacyEncoder uses quantization + hash-based encoding.


In [ ]:
# Extract numerical features
num_idx = [feature_names.index(f) for f in numerical_features]
X_num_fast = X_sample[:, num_idx]

print(f"Original numerical features ({sample_size} samples, {len(numerical_features)} features):")
print("Note: Already standardized (mean=0, std=1)")
display(pd.DataFrame(X_num_fast, columns=numerical_features).round(3))

# Fast Privacy Encoder
encoder_fast_num = FastPrivacyEncoder(
    encoding_dim=4,
    polynomial_degree=1,
    noise_level=0.1,
    use_mixing=True,
    hash_seed=42
)
X_num_fast_encoded = encoder_fast_num.fit_transform(X_num_fast)

print(f"\nAfter FastPrivacyEncoder:")
print(f"Shape: {X_num_fast.shape} → {X_num_fast_encoded.shape} (expansion: {X_num_fast_encoded.shape[1]/X_num_fast.shape[1]:.1f}x)")
print("Note: Continuous values are quantized, then hash-based encoded")
display(pd.DataFrame(X_num_fast_encoded, columns=[f"fast_{i}" for i in range(X_num_fast_encoded.shape[1])]).round(3))


### 5.4 Fast Privacy Encoder on All Features Together

FastPrivacyEncoder automatically detects feature types (discrete vs continuous) and handles them appropriately.


In [ ]:
print(f"Original all features ({sample_size} samples, {len(feature_names)} features):")
display(df_sample.round(3))

# Fast Privacy Encoder on all features
encoder_fast_all = FastPrivacyEncoder(
    encoding_dim=4,
    polynomial_degree=1,
    noise_level=0.1,
    use_mixing=True,
    hash_seed=42
)
X_all_fast_encoded = encoder_fast_all.fit_transform(X_sample)

print(f"\nAfter FastPrivacyEncoder:")
print(f"Shape: {X_sample.shape} → {X_all_fast_encoded.shape} (expansion: {X_all_fast_encoded.shape[1]/X_sample.shape[1]:.1f}x)")
print(f"\nDetected feature types:")
print(f"  Discrete features (hash tables): {encoder_fast_all.discrete_features}")
print(f"  Continuous features (quantization): {encoder_fast_all.continuous_features}")
print(f"\nNote: Automatically handles binary, categorical, and continuous features together!")
print(f"  - Binary/Categorical → Hash-based lookup tables")
print(f"  - Continuous → Quantization + hash-based encoding")
display(pd.DataFrame(X_all_fast_encoded, columns=[f"fast_{i}" for i in range(X_all_fast_encoded.shape[1])]).round(3))


## Example 4: All Features Together (Mixed Types)

Shows how all feature types (binary, categorical, numerical) work together in the same pipeline.


In [ ]:
print(f"Original all features ({sample_size} samples, {len(feature_names)} features):")
display(df_sample.round(3))


### 4.1 Encoding: Chaotic System (all features)


In [ ]:
encoder_all = ChaoticSystemEncoder(map_type='logistic', n_iterations=5)
X_all_encoded = encoder_all.fit_transform(X_sample)

print(f"After encoding: shape {X_sample.shape} → {X_all_encoded.shape}")
display(pd.DataFrame(X_all_encoded, columns=[f"{f}_enc" for f in feature_names]).round(3))


### 4.2 Mixing: Quantum Entanglement (all features)


In [ ]:
mixer_all = QuantumEntanglementEncoder(entanglement_strength=0.4)
mixer_all.fit(X_all_encoded)
X_all_mixed = mixer_all.transform(X_all_encoded)

print(f"After mixing: shape {X_all_mixed.shape}")
print("Note: All features (binary, categorical, numerical) are mixed together")
display(pd.DataFrame(X_all_mixed, columns=[f"mixed_{i}" for i in range(X_all_mixed.shape[1])]).round(3))


### 4.3 Pipeline: Encode then Mix (all features)


In [ ]:
pipeline_all = EncodeThenMixPipeline(
    ChaoticSystemEncoder(map_type='logistic', n_iterations=5),
    QuantumEntanglementEncoder(entanglement_strength=0.4)
)
X_all_pipeline = pipeline_all.fit_transform(X_sample)

print(f"After pipeline: shape {X_all_pipeline.shape}")
display(pd.DataFrame(X_all_pipeline, columns=[f"final_{i}" for i in range(X_all_pipeline.shape[1])]).round(3))


## Summary

### Key Takeaways:

1. **ENCODING (Irreversible Transformation)**:
   - `ChaoticSystemEncoder`: Transforms values through chaotic maps (logistic, henon)
   - `OrthogonalSubspaceEncoder`: Projects features into higher-dimensional orthogonal spaces
   - `FastPrivacyEncoder`: ✅ NEW - Hash-based encoding + polynomial expansion (like FHE but faster)
   - Works on: Binary, Categorical, Numerical (all types)

2. **MIXING (Feature Combination)**:
   - `ClassicalXORMixer`: Simple circular XOR/additive mixing
   - `QuantumEntanglementEncoder`: Non-linear quantum-inspired mixing with correlations
   - `QuantumWalkEncoder`: Graph-based quantum walk (expands dimensions)
   - Works on: Binary, Categorical, Numerical (all types)

3. **PIPELINE (Encode + Mix)**:
   - `EncodeThenMixPipeline`: First encode, then mix
   - Provides both privacy (encoding) and feature mixing (mixing)
   - Can be applied to any data type or combination

4. **FAST PRIVACY ENCODER (NEW)**:
   - ✅ **Hash-based lookup tables** for discrete values (binary/categorical)
   - ✅ **Quantization + hash** for continuous values
   - ✅ **Polynomial feature expansion** preserves relationships for ML
   - ✅ **Automatic feature type detection** (discrete vs continuous)
   - ✅ **Fast computation** (hash lookups vs cryptographic operations)
   - ⚠️ **Trade-off**: Optimized for utility (preserves information), weaker privacy

5. **DATA TYPE HANDLING**:
   - **Binary**: Hash-based encoding (FastPrivacy) or 0/1 → continuous [0,1] (Chaotic)
   - **Categorical**: Hash tables (FastPrivacy) or normalized → encoded (others)
   - **Numerical**: Quantization + hash (FastPrivacy) or standardized → encoded (others)
   - **All types work together** in the same pipeline!
